In [2]:
"""
Generate a grouped Geneva Emotion Wheel (GEW) visualization.
Shows the 8 emotion clusters projected onto the unit circle.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ── Emotion grouping (same as in RL scripts) ──
EMOTION_GROUPS = {
    "happy":       ["happy", "joy"],
    "pleasant":    ["love", "lovely", "sentimental"],
    "exciting":    ["exciting", "fun"],
    "calm":        ["mellow"],
    "sad":         ["sad", "melancholy"],
    "negative":    ["depressing"],
    "anger":       ["hate", "terrible"],
    "fear":        ["shock"],
}

# ── Colors for each cluster ──
COLORS = {
    "happy":     "#FFD700",   # gold
    "pleasant":  "#FF69B4",   # hot pink
    "exciting":  "#FF4500",   # orange-red
    "calm":      "#87CEEB",   # sky blue
    "sad":       "#4169E1",   # royal blue
    "negative":  "#708090",   # slate gray
    "anger":     "#DC143C",   # crimson
    "fear":      "#9932CC",   # dark orchid
}


def load_emotion_centers_grouped(csv_path):
    """Load and compute grouped emotion centers (identical to RL scripts)."""
    df = pd.read_csv(csv_path)

    def normalize_col(col):
        return 2 * (col - col.min()) / (col.max() - col.min()) - 1

    df["Valence_norm"] = normalize_col(df["Valence"])
    df["Arousal_norm"] = normalize_col(df["Arousal"])

    grouped_centers = {}
    raw_points = {}

    for group, emotions in EMOTION_GROUPS.items():
        sub = df[df["Emotion"].isin(emotions)]
        if sub.empty:
            continue

        v = sub["Valence_norm"].mean()
        a = sub["Arousal_norm"].mean()

        # Store raw (pre-normalization) center
        raw_points[group] = (v, a)

        # Project onto unit circle
        vec = np.array([v, a])
        vec = vec / (np.linalg.norm(vec) + 1e-9)
        grouped_centers[group] = tuple(vec)

    return grouped_centers, raw_points


def plot_grouped_gew(centers, raw_points, save_path):
    """Create a publication-quality grouped GEW figure."""

    fig, ax = plt.subplots(1, 1, figsize=(8, 8), facecolor="white")

    # ── Draw unit circle ──
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), color="#CCCCCC", linewidth=1.5,
            linestyle="--", zorder=1, label="Unit circle")

    # ── Draw axes ──
    ax.axhline(0, color="#E0E0E0", linewidth=0.8, zorder=0)
    ax.axvline(0, color="#E0E0E0", linewidth=0.8, zorder=0)

    # ── Plot each cluster ──
    for name, (cx, cy) in centers.items():
        color = COLORS[name]
        rx, ry = raw_points[name]

        # Draw arrow from origin to unit-circle point
        ax.annotate(
            "", xy=(cx, cy), xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=2, alpha=0.6),
            zorder=2,
        )

        # Draw the raw center as a small "x"
        ax.scatter(rx, ry, marker="x", color=color, s=60, linewidths=1.5,
                   zorder=3, alpha=0.7)

        # Draw the projected point as a large dot on the circle
        ax.scatter(cx, cy, color=color, s=220, edgecolors="black",
                   linewidths=1.2, zorder=4)

        # Label — offset outward from the circle for readability
        offset = 0.15
        lx = cx * (1 + offset)
        ly = cy * (1 + offset)
        ax.text(lx, ly, name.capitalize(),
                fontsize=12, fontweight="bold", color=color,
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                          edgecolor=color, alpha=0.85),
                zorder=5)

    # ── Axis labels ──
    ax.set_xlabel("Valence", fontsize=14, fontweight="bold")
    ax.set_ylabel("Arousal", fontsize=14, fontweight="bold")
    ax.set_title("Grouped Geneva Emotion Wheel (8 Clusters)",
                 fontsize=15, fontweight="bold", pad=15)

    # ── Quadrant labels ──
    ql = dict(fontsize=10, alpha=0.4, ha="center", va="center", style="italic")
    ax.text( 0.7,  0.7, "High V, High A", **ql)
    ax.text(-0.7,  0.7, "Low V, High A", **ql)
    ax.text(-0.7, -0.7, "Low V, Low A", **ql)
    ax.text( 0.7, -0.7, "High V, Low A", **ql)

    # ── Legend ──
    legend_items = []
    legend_items.append(mpatches.Patch(color="none", label="● = projected center"))
    legend_items.append(mpatches.Patch(color="none", label="✕ = raw mean center"))
    ax.legend(handles=legend_items, loc="lower right", fontsize=10,
              framealpha=0.9, edgecolor="#CCCCCC")

    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.45, 1.45)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.15)
    ax.tick_params(labelsize=11)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Saved: {save_path}")


if __name__ == "__main__":
    csv_path = os.path.join("data", "Metacsv", "emotion_summary.csv")
    out_dir  = "results_compare"
    os.makedirs(out_dir, exist_ok=True)

    centers, raw_pts = load_emotion_centers_grouped(csv_path)

    print("Grouped emotion centers (unit circle):")
    for name, (v, a) in centers.items():
        angle = np.degrees(np.arctan2(a, v))
        print(f"  {name:12s}  [{v:+.4f}, {a:+.4f}]  angle={angle:6.1f}°")

    save_path = os.path.join(out_dir, "grouped_gew.png")
    plot_grouped_gew(centers, raw_pts, save_path)

Grouped emotion centers (unit circle):
  happy         [+0.9051, +0.4253]  angle=  25.2°
  pleasant      [+0.3880, -0.9217]  angle= -67.2°
  exciting      [+0.6821, +0.7312]  angle=  47.0°
  calm          [-0.1468, -0.9892]  angle= -98.4°
  sad           [-0.9501, -0.3119]  angle=-161.8°
  negative      [-0.7414, -0.6711]  angle=-137.8°
  anger         [-0.9402, +0.3406]  angle= 160.1°
  fear          [-0.5416, +0.8406]  angle= 122.8°
  Saved: results_compare/grouped_gew.png
